# Fusion products and extensions

This notebook introduces two constructions on modules over a vertex algebra $V$: the fusion product $M \boxtimes N$ of two modules, computed by the Nahm–Gaberdiel–Kausch algorithm [Nah1994, GK1996], and the extensions between modules, computed from projective covers. Every example is small and has a known answer: the Lee–Yang and Ising minimal models, affine $\mathfrak{sl}_2$ at level 1, lattice vertex algebras, the first logarithmic fusion product at $c = -2$, and a Jordan block of $L_0$ on a Virasoro module.

| method | what it gives | section |
|---|---|---|
| `M.fusion_product(N)` | the fusion product $P = M \boxtimes N$, computed lazily | 1 |
| `P.depth_quotient_up_to(d, max_depth)` | the depth-$d$ quotient of $P$, with `l0_matrix()`, `conformal_weights()`, `mode_matrix(a, n)` and `zhu_algebra_module(A)` | 1–4 |
| `F.special_subspace_dimensions()` | the dimensions of Nahm's special subspace, as far as the window sees it | 2 |
| `L.projective_cover(max_weight, max_depth)` | the projective cover $\pi: P \to L$ of a simple module over a quotient | 5 |
| `M.ext1_up_to(N, depth, cover)` | $\operatorname{Ext}^1(M, N)$, with `extension(c)` for the extension of a class $c$ | 5, 6 |
| `M.radical_series_up_to(depth, simples)`, `M.socle_series_up_to(depth, simples)` | the Loewy layers of $M$ | 5, 6 |

**Prerequisites:** `01_getting_started.ipynb`, `09_zhu_algebras.ipynb`, `10_modules.ipynb`.

**Elsewhere:** the research examples come afterwards: the triplet algebra $\mathcal W(2)$, with its projective covers, extensions and fusion rules, in `15_triplet_modules.ipynb`, and Virasoro fusion at $c = 0$ in `16_virasoro_c0_fusion.ipynb`.

**Contents**

1. Fusion by the Nahm–Gaberdiel–Kausch algorithm
2. The window
3. More rational examples: Ising and affine $\mathfrak{sl}_2$ at level 1
4. A logarithmic fusion product at $c = -2$
5. Projective covers and $\operatorname{Ext}^1$: the semisimple case
6. A non-split extension: a Jordan block of $L_0$
7. Limitations

## 1. Fusion by the Nahm–Gaberdiel–Kausch algorithm

The fusion product $M \boxtimes N$ of two $V$-modules is a quotient of $M \otimes N$ on which $V$ acts through a coproduct. The library uses the conventions of Kanade and Ridout [KR2019]: the module $M$ is inserted at $w = 1$ and $N$ at $w = 0$, and the mode $v_{(k)}$ of $v \in V$ acts by

$$\Delta\bigl(v_{(k)}\bigr) = \sum_{j \ge 0} \binom{k}{j}\, v_{(j)} \otimes 1 + 1 \otimes v_{(k)}.$$

For the Virasoro field this gives $\Delta(L_0) = L_{-1} \otimes 1 + L_0 \otimes 1 + 1 \otimes L_0$. Since $M \boxtimes N$ is infinite-dimensional, the library computes its *depth-$d$ quotient* $(M \boxtimes N)^{(d)}$ [Nah1994, GK1996]: the quotient of $M \boxtimes N$ by the span of all words of modes of total degree $> d$, where the *degree* of a mode is the amount by which it raises the conformal weight. The depth-$d$ quotient is finite-dimensional, and $\Delta(L_0)$ and the modes of degree $0, \dots, d$ act on it. For a direct sum of highest-weight modules, the depth-$d$ quotient is the sum of the weight spaces of depth $\le d$ of the summands, each depth counted from the lowest weight of its summand.

The Lee–Yang model is the Virasoro minimal model of central charge $c = -\frac{22}{5}$, with two simple modules, of lowest weights $0$ and $-\frac15$ [BPZ1984]. With $\varphi = L(-\frac15)$ and $1 = L(0)$ the vacuum module, its fusion rule is $\varphi \boxtimes \varphi = 1 + \varphi$. In the cell below, the modules are the simple quotients of Verma modules of the universal Virasoro vertex algebra at this central charge. `M.fusion_product(N)` builds the product without computing anything yet, and `depth_quotient_up_to(d, max_depth=K)` computes the depth-$d$ quotient from the tensors of total depth $\le K$ (section 2). The cell prints the dimension and a basis of the depth-0 quotient of $\varphi \boxtimes \varphi$.

In [1]:
import vertex_algebras as va
from vertex_algebras import *

V = va.Virasoro(QQ, -22/5, conformal_vector='L')
L = V.gen('L')
phi = V.verma_module(-1/5).simple_quotient()
one = V.verma_module(0).simple_quotient()
P = phi.fusion_product(phi)
F = P.depth_quotient_up_to(0, max_depth=3)
F.dimension(), F.basis()

(2, [(v, v), (L_{-1} v, v)])

The basis is given by representatives $x \otimes y$, here $v \otimes v$ and $L_{-1}v \otimes v$, where $v$ is the lowest-weight vector of $\varphi$. `l0_matrix()` is the matrix of $\Delta(L_0)$, whose columns are the images of the basis vectors. This matrix is not diagonal, because $\Delta(L_0)$ contains $L_{-1} \otimes 1$. `conformal_weights()` lists the eigenvalues $h$ of $\Delta(L_0)$, each with the sizes of its Jordan blocks. The cell below shows one state of weight $-\frac15$ and one of weight $0$: the lowest-weight vectors of $\varphi$ and of $1$.

In [2]:
F.l0_matrix(), F.conformal_weights()

(
[ -2/5 -2/25]                         
[    1   1/5], [(-1/5, [1]), (0, [1])]
)

The depth-2 quotient contains the weight spaces of depth $\le 2$ of both summands. The module $\varphi$ has one state at each of the weights $-\frac15, \frac45, \frac95$, and the vacuum module has states at the weights $0$ and $2$ only, since $L_{-1}\Omega = 0$. The first cell below computes the conformal weights of the depth-2 quotient; the second prints the dimensions of the weight spaces of depth $0, 1, 2$ of $\varphi$ and of $1$ for comparison.

In [3]:
P.depth_quotient_up_to(2, max_depth=6).conformal_weights()

[(-1/5, [1]), (0, [1]), (4/5, [1]), (9/5, [1]), (2, [1])]

In [4]:
[phi.weight_space(-1/5 + d).dimension() for d in range(3)], [one.weight_space(d).dimension() for d in range(3)]

([1, 1, 1], [1, 0, 1])

The vacuum module is the unit for fusion, on either side. The cell below computes the depth-0 quotients of $1 \boxtimes \varphi$ and of $\varphi \boxtimes 1$:

In [5]:
one.fusion_product(phi).depth_quotient_up_to(0, 3).conformal_weights(), phi.fusion_product(one).depth_quotient_up_to(0, 3).conformal_weights()

([(-1/5, [1])], [(-1/5, [1])])

The depth-0 quotient is a module over the Zhu algebra $A(V)$ [Zhu1996], through the zero modes. `zhu_algebra_module(A)` returns the depth-0 quotient as such a module, which can then be passed to `induced_module` (see `10_modules.ipynb`, section 4). Other modes are available as matrices: `mode_matrix(a, n)` is the matrix of $\Delta(a_{(n)})$ and `zero_mode_matrix(a)` that of the zero mode of $a$, and `projection(d')` is the projection to the depth-$d'$ quotient, for $d' \le d$.

In [6]:
F.zhu_algebra_module(V.zhu_algebra())

Module of dimension 2 over Zhu algebra of Universal vertex algebra on generators ['L'] over Rational Field

## 2. The window

The computation only uses the tensors $x \otimes y$ of total depth $\le K$ = `max_depth`, where the depth of $x$ is its weight minus the lowest weight of $M$ (and likewise for $y$ and $N$), together with the relations whose terms all lie in this *window*. Nahm [Nah1994] and Gaberdiel–Kausch [GK1996] showed that the depth-$d$ quotient is spanned by the images of $M^{ss} \otimes N^{(d)}$, where $N^{(d)}$ is the depth-$d$ quotient of $N$ and

$$M^{ss} = M \Big/ \sum_{v} \sum_{b < 0} v_{(b)} M$$

is Nahm's *special subspace* of $M$; the outer sum runs over the strong generators $v$ of $V$ (see also [KR2019]). For the Virasoro algebra, $M^{ss} = M / \sum_{n \ge 2} L_{-n} M$. `special_subspace_dimensions()` gives the dimensions of $M^{ss}$ by depth, as far as the window sees them. For $\varphi$, the special subspace is spanned by the images of $v$ and $L_{-1}v$, since the null vector at depth 2 makes $L_{-1}^2 v$ a multiple of $L_{-2} v$. This explains the bound $\dim (\varphi \boxtimes \varphi)^{(0)} \le 2 \cdot 1$ of section 1.

In [7]:
F.special_subspace_dimensions()

{0: 1, 1: 1, 2: 0, 3: 0}

The result is reliable only once the window is large enough, and the code cannot tell when this is the case: the result is computed "up to" `max_depth`, and checking that it is stable as `max_depth` grows is left to the user. In a window that is too small, the dimension can be too small, if the special subspace does not yet fit in the window, or too large, if the null vectors of the factors do not yet fit in it (their relations are what cut the product down). Once the special subspace fits, the result maps onto the true depth-$d$ quotient, and it only shrinks as the window grows. For the Lee–Yang product $\varphi \boxtimes \varphi$ at depth 0, the cell below computes the dimension for $K = 0, \dots, 5$; it is too small only at $K = 0$.

In [8]:
[P.depth_quotient_up_to(0, K).dimension() for K in range(6)]

[1, 2, 2, 2, 2, 2]

In the Ising model ($c = \frac12$), $\varepsilon \boxtimes \varepsilon = 1$ for $\varepsilon = L(\frac12)$ [BPZ1984], so the depth-0 quotient has dimension 1. The first cell below computes it for $K = 0, \dots, 5$: in small windows the result is too large, and it is correct from $K = 3$ on. The reason is that the null vectors of $\varepsilon$ lie at depths 2 and 3. The second cell shows that the maximal submodule of the Verma module $M(\frac12)$ has dimensions $0, 0, 1, 2$ at the depths $0, \dots, 3$, so it is generated by a singular vector at depth 2 and a further generator at depth 3.

In [9]:
Vi = va.Virasoro(QQ, 1/2, conformal_vector='L')
eps = Vi.verma_module(1/2).simple_quotient()
[eps.fusion_product(eps).depth_quotient_up_to(0, K).dimension() for K in range(6)]

[1, 2, 2, 1, 1, 1]

In [10]:
[Vi.verma_module(1/2).maximal_submodule().weight_space(1/2 + d).dimension() for d in range(4)]

[0, 0, 1, 2]

## 3. More rational examples: Ising and affine $\mathfrak{sl}_2$ at level 1

The three simple modules of the Ising model have lowest weights $0, \frac12, \frac1{16}$, and are written $1, \varepsilon, \sigma$ [BPZ1984]. Their fusion rules are $\varepsilon \boxtimes \varepsilon = 1$, $\varepsilon \boxtimes \sigma = \sigma$ and $\sigma \boxtimes \sigma = 1 + \varepsilon$. The cell below tabulates the conformal weights of the depth-0 quotients of all the products (rows and columns $1, \varepsilon, \sigma$); every fusion rule can be read off the table.

In [11]:
ising = [Vi.verma_module(h).simple_quotient() for h in (0, 1/2, 1/16)]
def weights(A, B):
    return [h for h, sizes in A.fusion_product(B).depth_quotient_up_to(0, max_depth=3).conformal_weights()]
table([[weights(A, B) for B in ising] for A in ising], header_row=['1', 'ε', 'σ'], header_column=['', '1', 'ε', 'σ'])

,1,ε,σ
1,\(\left[0\right]\),\(\left[\frac{1}{2}\right]\),\(\left[\frac{1}{16}\right]\)
ε,\(\left[\frac{1}{2}\right]\),\(\left[0\right]\),\(\left[\frac{1}{16}\right]\)
σ,\(\left[\frac{1}{16}\right]\),\(\left[\frac{1}{16}\right]\),"\(\left[0, \frac{1}{2}\right]\)"


The depth-1 quotient of $\sigma \boxtimes \sigma$ contains, in addition, the state $L_{-1}w$ of weight $\frac32$, where $w$ is the lowest-weight vector of $\varepsilon$; the vacuum module has no state of depth 1:

In [12]:
sigma = ising[2]
sigma.fusion_product(sigma).depth_quotient_up_to(1, max_depth=4).conformal_weights()

[(0, [1]), (1/2, [1]), (3/2, [1])]

Affine $\mathfrak{sl}_2$ at level 1 has two simple modules: the vacuum module $L(0)$ and the module $L(\omega_1)$, whose top space is the doublet, of conformal weight $\frac14$. These are the two modules $V_{\mathbb Z\alpha}$ and $V_{\alpha/2 + \mathbb Z\alpha}$ of the lattice vertex algebra of $A_1$ (`10_modules.ipynb`, sections 2 and 5), and the lattice fusion rule $V_{\lambda+L} \boxtimes V_{\mu+L} = V_{\lambda+\mu+L}$ predicts $L(\omega_1) \boxtimes L(\omega_1) = L(0)$. The library returns the lattice answer directly (see the end of this section). As an independent check, this section first computes the product with the Nahm–Gaberdiel–Kausch algorithm from the affine description. The cell below constructs the two modules as the simple quotients of the modules induced from the trivial module and from the doublet of the Zhu algebra $U(\mathfrak{sl}_2)$ [FZ1992], and prints the character of $L(\omega_1)$.

In [13]:
A1 = va.Affine(QQ, ['A', 1], 1, names=['E', 'H', 'F'], conformal_vector='sugawara')
Z = A1.zhu_algebra()
doublet = Z.module({'E': matrix([[0, 1], [0, 0]]), 'H': matrix([[1, 0], [0, -1]]), 'F': matrix([[0, 0], [1, 0]])})
trivial = Z.module({'E': matrix(QQ, 1), 'H': matrix(QQ, 1), 'F': matrix(QQ, 1)})
D = A1.induced_module(doublet).simple_quotient()
O = A1.induced_module(trivial).simple_quotient()
D.character(4)

2*q^(1/4) + 2*q^(5/4) + 6*q^(9/4) + 8*q^(13/4) + O(q^4)

The depth-0 quotient of $L(\omega_1) \boxtimes L(\omega_1)$ is one state of weight 0, on which $H_0$ acts by 0: the top space of $L(0)$. The depth-1 quotient adds three states of weight 1. `horizontal_grades()` gives the $\mathfrak{sl}_2$-weights of the basis states, in fundamental weight coordinates: $2, 0, 0, -2$. These are the weight $0$ of the state of weight 0 and the weights $2, 0, -2$ of the adjoint representation, which spans the weight-1 space of $L(0)$, with basis $E_{-1}\Omega, H_{-1}\Omega, F_{-1}\Omega$.

In [14]:
DD = D.fusion_product(D)
F0 = DD.depth_quotient_up_to(0, max_depth=3)
F1 = DD.depth_quotient_up_to(1, max_depth=4)
F0.conformal_weights(), F0.zero_mode_matrix(A1.gen('H')), F1.conformal_weights(), F1.horizontal_grades()

([(0, [1])], [0], [(0, [1]), (1, [1, 1, 1])], [(2,), (0,), (0,), (-2,)])

The vacuum module is again the unit: the depth-0 quotient of $L(0) \boxtimes L(\omega_1)$ is the doublet top space of $L(\omega_1)$, at weight $\frac14$.

In [15]:
O.fusion_product(D).depth_quotient_up_to(0, max_depth=3).conformal_weights()

[(1/4, [1, 1])]

**Lattice vertex algebras.** The library's implementation of the Nahm–Gaberdiel–Kausch algorithm needs the vertex algebra to be given by finitely many strong generators and their OPEs, and lattice vertex algebras are not given in this way. For them the answer is known: $V_{\lambda+L} \boxtimes V_{\mu+L} = V_{\lambda+\mu+L}$ for every non-degenerate integral lattice [DL1993], and `M.fusion_product(N)` returns this module. `module()` is the product itself, and `depth_quotient_up_to(d)` gives its depth-$d$ quotients, exactly and with the same interface as above. For the $A_1$ lattice with the Sugawara vector $\frac14{:}JJ{:}$, the two simple modules are $L(0)$ and $L(\omega_1)$ again. The cell below computes $V_{\alpha/2 + L} \boxtimes V_{\alpha/2 + L}$, and checks that its depth-1 quotient has the conformal weights computed above from the affine description.

In [16]:
A = va.Lattice(QQ, matrix([[2]])); J = A.J(0)
A = A.with_conformal_vector(J.no(J) / 4)
Dl = A.simple_module([1/2])
P = Dl.fusion_product(Dl)
P.module(), P.depth_quotient_up_to(1).conformal_weights() == F1.conformal_weights()

(Module V_(lambda+L), lambda = (0), over Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field,
 True)

Fusion distributes over direct sums. For the $A_2$ lattice, the fusion products of the three simple modules $V_0, V_1, V_2$ follow the group law of $L^\circ/L \cong \mathbb Z/3$. The first cell below prints the coset representatives of all the products. The second prints the conformal weights of the depth-0 quotient of $V_1 \boxtimes (V_1 \oplus V_2)$, where $V_1$ and $V_2$ are the two modules of lowest weight $\frac13$: one state of weight 0 from $V_1 \boxtimes V_2 = V_0$ and the three-dimensional top space of $V_1 \boxtimes V_1 = V_2$.

In [17]:
A2 = va.Lattice(QQ, matrix([[2, -1], [-1, 2]])); J0, J1 = A2.J(0), A2.J(1)
A2 = A2.with_conformal_vector((2*J0.no(J0) + J0.no(J1) + J1.no(J0) + 2*J1.no(J1)) / 6)
V0, V1, V2 = A2.simple_modules()
[[X.fusion_product(Y).module().coset_representative() for Y in (V0, V1, V2)] for X in (V0, V1, V2)]

[[(0, 0), (1/3, 2/3), (2/3, 1/3)],
 [(1/3, 2/3), (2/3, 1/3), (0, 0)],
 [(2/3, 1/3), (0, 0), (1/3, 2/3)]]

In [18]:
V1.fusion_product(V1.direct_sum(V2)).depth_quotient_up_to(0).conformal_weights()

[(0, [1]), (1/3, [1, 1, 1])]

## 4. A logarithmic fusion product at $c = -2$

The same algorithm finds fusion products on which $L_0$ is not diagonalisable. Gaberdiel and Kausch [GK1996] (see also [KR2019, section 8]) fused the Virasoro module $L(-\frac18)$ at $c = -2$ with itself: the depth-0 quotient is two-dimensional, with a single Jordan block of $L_0$ at weight 0. The cell below prints the basis of the depth-0 quotient, the matrix of $\Delta(L_0)$ and the conformal weights.

In [19]:
Vc = va.Virasoro(QQ, -2, conformal_vector='L')
mu = Vc.verma_module(-1/8).simple_quotient()
Fc = mu.fusion_product(mu).depth_quotient_up_to(0, max_depth=3)
Fc.basis(), Fc.l0_matrix(), Fc.conformal_weights()

(
                         [ -1/4 -1/16]            
[(v, v), (L_{-1} v, v)], [    1   1/4], [(0, [2])]
)

The output `[(0, [2])]` reads: the generalised eigenvalue 0, with one Jordan block of size 2. Here $L(-\frac18)$ is a module over the universal Virasoro algebra. The triplet algebra $\mathcal W(2)$ at $c = -2$ has a simple module $\mathcal W(-\frac18)$ of the same lowest weight, and $\mathcal W(-\frac18) \boxtimes \mathcal W(-\frac18)$ is the projective cover $\mathcal P(0)$ of the vacuum module of $\mathcal W(2)$ [GK1996b]. This example, with the whole fusion table of $\mathcal W(2)$, is in `15_triplet_modules.ipynb`, section 9. The Virasoro fusion products at $c = 0$, with Jordan blocks of rank 2 and 3 and their logarithmic couplings, are in `16_virasoro_c0_fusion.ipynb`.

## 5. Projective covers and $\operatorname{Ext}^1$: the semisimple case

The modules of this section are positive-energy modules over a quotient $Q$ of a universal vertex algebra, whose generalised $L_0$-eigenvalues lie in $h + \mathbb Z_{\ge 0}$ for a fixed $h$. For such a module $N$, the *degree-$n$ space* $N(n)$, of generalised weight $h + n$, is a module over the level-$n$ Zhu algebra $A_n(Q)$ [DLM1998]. For a module $U$ over $A_n(Q)$, let $\bar M_n(U)$ be the universal module induced from $U$ placed in degree $n$. Then

$$\operatorname{Hom}_Q(\bar M_n(U), N) = \operatorname{Hom}_{A_n(Q)}(U, N(n))$$

[DLM1998, He2017, BVY2019]. Hence, if $U$ is a projective $A_n(Q)$-module, then $\bar M_n(U)$ is projective in the category of these positive-energy modules.

For a simple module $L$ of lowest weight $h_L$, `L.projective_cover(max_weight, max_depth)` builds the projective cover $\pi: \bar M_n(A_n e) \to L$, where $e$ is a primitive idempotent of $A_n(Q)$ that acts nontrivially on the top space of $L$. The level $n$ is chosen so that $h_L - n$ is the lowest weight in $h_L + \mathbb Z$ among the simple modules found from the Zhu algebra. Then every module of finite length whose composition factors are simple $Q$-modules with lowest weights in $h_L + \mathbb Z$ lies in the category, with $h = h_L - n$.

Everything is computed "up to": `max_weight` bounds the weight of the relations of $Q$ that are used (the Zhu algebras computed from them map onto the true ones), and `max_depth` bounds the depth to which modules are computed. The first example is the Lee–Yang vertex algebra $Q = V / \langle {:}LL{:} - \frac3{10}\partial^2 L \rangle$. `Q.simple_modules_up_to(max_weight, max_depth)` returns its simple modules, the simple quotients of the modules induced from the simple modules of its Zhu algebra (`10_modules.ipynb`, section 8): again $L(-\frac15)$ and $L(0)$. The cell below prints their lowest weights (`top_weight()`) and the dimensions of their weight spaces of depth $0, \dots, 5$.

In [20]:
Q = V / [L.no(L) - 3/10*L.derivative(2)]
simples = Q.simple_modules_up_to(6, 5)
[S.top_weight() for S in simples], [[S.weight_space(S.top_weight() + d).dimension() for d in range(6)] for S in simples]

([-1/5, 0], [[1, 1, 1, 1, 2, 2], [1, 0, 1, 1, 1, 1]])

The Lee–Yang vertex algebra is rational [Wan1993], so every module in the category is projective, and each simple module is its own projective cover. The library finds this: in the cell below, the domains of the projective covers have the same weight space dimensions as the simple modules (up to depth 5), and each $\pi$ is surjective up to depth 5.

In [21]:
covers = [S.projective_cover(6, 5) for S in simples]
[[pi.domain().weight_space(S.top_weight() + d).dimension() for d in range(6)] for pi, S in zip(covers, simples)], [pi.is_surjective_up_to(5) for pi in covers]

([[1, 1, 1, 1, 2, 2], [1, 0, 1, 1, 1, 1]], [True, True])

For a surjection $\pi: P \to M$ with $P$ projective and kernel $K = \ker \pi$, the exact sequence

$$0 \to \operatorname{Hom}(M, N) \to \operatorname{Hom}(P, N) \to \operatorname{Hom}(K, N) \to \operatorname{Ext}^1(M, N) \to 0$$

[Wei1994] gives $\operatorname{Ext}^1(M, N) = \operatorname{Hom}(K, N) / \{f|_K : f \in \operatorname{Hom}(P, N)\}$. `M.ext1_up_to(N, depth, cover)` computes this quotient from the generators of $K$ up to the depth `depth` and the linear maps that commute with the modes in the degrees $\le$ `depth`. Relations of $K$ in higher degrees are not seen, which can make the result too large, and generators of $K$ in higher degrees are not seen either; so the stability of the result in `depth` must be checked. For Lee–Yang, every $\operatorname{Ext}^1$ between simple modules vanishes, as the cell below shows (rows: $M$, columns: $N$).

In [22]:
[[M.ext1_up_to(N, 4, pi).dimension() for N in simples] for M, pi in zip(simples, covers)]

[[0, 0], [0, 0]]

The *Loewy layers* of a module $X$ are the layers of its radical series, head first, and of its socle series, socle first; here $\operatorname{rad} X$ is the intersection of the kernels of all maps from $X$ to simple modules. `radical_series_up_to(depth, simples)` and `socle_series_up_to(depth, simples)` give each layer as the multiplicities of the modules in `simples`. The maps are found up to the depth `depth`, so the stability in `depth` must be checked. A direct sum of simple modules is its own head and its own socle, so both series have a single layer, as the cell below shows.

In [23]:
S = simples[0].direct_sum(simples[1])
S.radical_series_up_to(4, simples), S.socle_series_up_to(4, simples)

([[1, 1]], [[1, 1]])

## 6. A non-split extension: a Jordan block of $L_0$

The smallest non-split example of this notebook uses the universal Virasoro algebra at $c = \frac37$. There, the Verma module $M(2)$ is simple: the Kac determinant [Kac1979] has no zero at $c = \frac37$, $h = 2$, and the code finds no maximal submodule up to depth 6. The Zhu algebra is $\mathbb C[x]$, $x = [L]$ [FZ1992]. Let $A_0 = \mathbb C[x]/((x-2)^2)$, and let $U = A_0$ be its regular module, on which $[L]$ acts by a Jordan block. Since $U$ is projective over $A_0$, the induced module $P = \bar M(U)$ (`V.induced_module(U)`) is projective in the category of positive-energy modules with weights in $2 + \mathbb Z_{\ge 0}$ whose weight-2 space is an $A_0$-module, that is, on whose weight-2 space $(L_0 - 2)^2 = 0$.

The first cell below builds $P$ and $M(2)$, and checks that the maximal submodule of $M(2)$ vanishes up to depth 6. The second shows that $L_0$ acts by a Jordan block on the top space of $P$, with basis $u_0, u_1$.

In [24]:
V37 = va.Virasoro(QQ, 3/7, conformal_vector='L')
L37 = V37.gen('L')
A = V37.zhu_algebra(); x = A.gen(0)
A0 = A.quotient([(x - 2)^2])
P37 = V37.induced_module(A0.module({'L': matrix([[2, 1], [0, 2]])}))
M2 = V37.verma_module(2)
v = M2.highest_weight_vector()
[M2.maximal_submodule().weight_space(2 + d).dimension() for d in range(7)]

[0, 0, 0, 0, 0, 0, 0]

In [25]:
u0, u1 = P37.weight_space(2).basis()
L37.mode(u0, 0), L37.mode(u1, 0)

(2*u[0], u[0] + 2*u[1])

Module morphisms (`10_modules.ipynb`, section 10) show the structure of $P$. Let $v$ be the highest-weight vector of $M(2)$. The spaces $\operatorname{Hom}(P, M(2))$ and $\operatorname{Hom}(M(2), P)$ are one-dimensional, which the unpackings `[pi] = ...` and `[j] = ...` in the first cell below check: $\pi$ sends $u_1 \mapsto v$ and $u_0 \mapsto 0$, and $j$ sends $v \mapsto u_0$. The second cell shows that the kernel of $\pi$ and the image of $j$ have the same weight spaces, so $P$ is an extension of $M(2)$ by $M(2)$, and that $\operatorname{End}(P)$ is two-dimensional, spanned by the identity and by $j \circ \pi$, which acts on the top space as $L_0 - 2$.

In [26]:
[pi] = Hom(P37, M2).basis()
[j] = Hom(M2, P37).basis()
pi(u1), pi(u0), j(v)

(v, 0, u[0])

In [27]:
[pi.kernel().weight_space(2 + d).dimension() for d in range(5)], [j.image().weight_space(2 + d).dimension() for d in range(5)], len(End(P37).basis())

([1, 1, 2, 3, 5], [1, 1, 2, 3, 5], 2)

The Loewy layers of $P$ are $M(2) \mid M(2)$ (first cell below), and $\operatorname{Ext}^1(M(2), M(2))$, computed from the cover $\pi$, is one-dimensional in this category (second cell). `kernel_generators()` returns the generators of $\ker\pi$, and `basis()` gives each class by the images of these generators. The dimension is the same for `depth` 2, 3 and 4.

In [28]:
P37.radical_series_up_to(3, [M2]), P37.socle_series_up_to(3, [M2])

([[1], [1]], [[1], [1]])

In [29]:
E = M2.ext1_up_to(M2, 3, pi)
E.kernel_generators(), E.basis(), [M2.ext1_up_to(M2, d, pi).dimension() for d in (2, 3, 4)]

([u[0]], [(v,)], [1, 1, 1])

`E.extension(c, maps=True)` builds the extension module $X$ of a class $c$, the pushout of the cover along the map $\ker\pi \to N$ that represents $c$, together with the maps $N \to X \to M$. The extension module is a quotient of the direct sum $N \oplus P = M(2) \oplus P$, and its vectors print as `[0] ...` and `[1] ...`, their components in the two summands. For the basis class, the first cell below shows that $X$ has the weight spaces of $P$, a Jordan block of $L_0$ on its top space (so the extension does not split), and the same Loewy layers. As $\operatorname{Ext}^1$ is one-dimensional, $X$ is isomorphic to $P$. The second cell applies the two maps.

In [30]:
X, iota, proj = E.extension([1], maps=True)
a, b = X.weight_space(2).basis()
[X.weight_space(2 + d).dimension() for d in range(4)], [P37.weight_space(2 + d).dimension() for d in range(4)], (L37.mode(a, 0), L37.mode(b, 0)), X.socle_series_up_to(3, [M2])

([2, 2, 4, 6], [2, 2, 4, 6], (2*[0] v, [0] v + 2*[1] u[1]), [[1], [1]])

In [31]:
iota(v), proj(b)

([0] v, v)

The logarithmic analogue of this section, for a vertex algebra with finitely many simple modules, is the triplet algebra $\mathcal W(2)$: its projective covers $\mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0)$ and $\mathcal P(1)$ (through the level-1 Zhu algebra), their Loewy layers, $\operatorname{Ext}^1$ between its simple modules, and fusion products of projective modules are in `15_triplet_modules.ipynb`, section 10.

## 7. Limitations

* **Depth quotients only.** Only the depth-$d$ quotients of $M \boxtimes N$ are computed, not $M \boxtimes N$ as a module. When the top space is enough, `zhu_algebra_module` and `induced_module` give a module with the same top space.
* **Windows.** The code cannot tell whether a window is large enough: the stability of the results in `max_depth` (for fusion), in `max_weight` and `max_depth` (for projective covers) and in `depth` (for $\operatorname{Ext}^1$ and Loewy layers) must be checked by the user. A result that is stable over a few consecutive windows can still be wrong (`16_virasoro_c0_fusion.ipynb`, section 1).
* **Scope of the fusion algorithm.** The Nahm–Gaberdiel–Kausch algorithm is not available for lattice vertex algebras (the library returns the known answer instead, section 3), for modules with infinite-dimensional weight spaces, or for twisted modules.
* **$\operatorname{Ext}$.** $\operatorname{Ext}^n$ for $n \ge 2$ is not computed. $\operatorname{Ext}^1$ needs a cover: a projective cover from `projective_cover`, over a quotient with finite-dimensional Zhu algebras, or a map that the user knows to be a projective cover. Covers that are not projective give only part of $\operatorname{Ext}^1$ directly (`15_triplet_modules.ipynb`, section 10).

## References

- [Nah1994] W. Nahm, *Quasi-rational fusion products*, Internat. J. Modern Phys. B 8 (1994), 3693–3702, [arXiv:hep-th/9402039](https://arxiv.org/abs/hep-th/9402039), [doi:10.1142/S0217979294001597](https://doi.org/10.1142/S0217979294001597). — the depth quotients and the special subspace (sections 1, 2).
- [GK1996] M. R. Gaberdiel, H. G. Kausch, *Indecomposable fusion products*, Nuclear Phys. B 477 (1996), 293–318, [arXiv:hep-th/9604026](https://arxiv.org/abs/hep-th/9604026), [doi:10.1016/0550-3213(96)00364-1](https://doi.org/10.1016/0550-3213(96)00364-1). — the Nahm–Gaberdiel–Kausch algorithm and the fusion product at $c = -2$ (sections 1, 2, 4).
- [KR2019] S. Kanade, D. Ridout, *NGK and HLZ: fusion for physicists and mathematicians*, in: Affine, vertex and W-algebras, Springer INdAM Ser. 37 (2019), 135–181, [arXiv:1812.10713](https://arxiv.org/abs/1812.10713), [doi:10.1007/978-3-030-32906-8_7](https://doi.org/10.1007/978-3-030-32906-8_7). — the conventions of the fusion computation (sections 1, 2, 4).
- [BPZ1984] A. A. Belavin, A. M. Polyakov, A. B. Zamolodchikov, *Infinite conformal symmetry in two-dimensional quantum field theory*, Nuclear Phys. B 241 (1984), 333–380, [doi:10.1016/0550-3213(84)90052-X](https://doi.org/10.1016/0550-3213(84)90052-X). — the Lee–Yang and Ising models (sections 1–3).
- [Zhu1996] Y. Zhu, *Modular invariance of characters of vertex operator algebras*, J. Amer. Math. Soc. 9 (1996), 237–302, [doi:10.1090/S0894-0347-96-00182-8](https://doi.org/10.1090/S0894-0347-96-00182-8). — the Zhu algebra (section 1).
- [FZ1992] I. B. Frenkel, Y. Zhu, *Vertex operator algebras associated to representations of affine and Virasoro algebras*, Duke Math. J. 66 (1992), 123–168, [doi:10.1215/S0012-7094-92-06604-X](https://doi.org/10.1215/S0012-7094-92-06604-X). — the Zhu algebras of affine and Virasoro vertex algebras (sections 3, 6).
- [DL1993] C. Dong, J. Lepowsky, *Generalized vertex algebras and relative vertex operators*, Progr. Math. 112, Birkhäuser (1993), [doi:10.1007/978-1-4612-0353-7](https://doi.org/10.1007/978-1-4612-0353-7). — the fusion rules of lattice vertex algebras (section 3).
- [GK1996b] M. R. Gaberdiel, H. G. Kausch, *A rational logarithmic conformal field theory*, Phys. Lett. B 386 (1996), 131–137, [arXiv:hep-th/9606050](https://arxiv.org/abs/hep-th/9606050), [doi:10.1016/0370-2693(96)00949-5](https://doi.org/10.1016/0370-2693(96)00949-5). — the fusion rules of the triplet algebra $\mathcal W(2)$ (section 4).
- [DLM1998] C. Dong, H. Li, G. Mason, *Vertex operator algebras and associative algebras*, J. Algebra 206 (1998), 67–96, [arXiv:q-alg/9612010](https://arxiv.org/abs/q-alg/9612010), [doi:10.1006/jabr.1998.7425](https://doi.org/10.1006/jabr.1998.7425). — the level-$n$ Zhu algebras, the universal modules $\bar M_n(U)$ and the adjunction (section 5).
- [He2017] X. He, *Higher level Zhu algebras are subquotients of universal enveloping algebras*, J. Algebra 491 (2017), 265–279, [arXiv:1712.05671](https://arxiv.org/abs/1712.05671), [doi:10.1016/j.jalgebra.2017.08.014](https://doi.org/10.1016/j.jalgebra.2017.08.014). — the model of the level-$n$ Zhu algebras used (section 5).
- [BVY2019] K. Barron, N. Vander Werf, J. Yang, *Higher level Zhu algebras and modules for vertex operator algebras*, J. Pure Appl. Algebra 223 (2019), 3295–3317, [arXiv:1710.04767](https://arxiv.org/abs/1710.04767), [doi:10.1016/j.jpaa.2018.11.002](https://doi.org/10.1016/j.jpaa.2018.11.002). — higher level Zhu algebras and their modules (section 5).
- [Wan1993] W. Wang, *Rationality of Virasoro vertex operator algebras*, Int. Math. Res. Not. 1993, no. 7, 197–211, [doi:10.1155/S1073792893000212](https://doi.org/10.1155/S1073792893000212). — the minimal models are rational (section 5).
- [Wei1994] C. A. Weibel, *An introduction to homological algebra*, Cambridge Stud. Adv. Math. 38, Cambridge University Press (1994), [doi:10.1017/CBO9781139644136](https://doi.org/10.1017/CBO9781139644136). — $\operatorname{Ext}^1$ from a projective cover (section 5).
- [Kac1979] V. G. Kac, *Contravariant form for infinite-dimensional Lie algebras and superalgebras*, in: Group theoretical methods in physics, Lecture Notes in Phys. 94, Springer (1979), 441–445. — the Kac determinant (section 6).